# Quantization - use fewer bits to represent numbers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/01_quantization/quantization.ipynb)

**Goal.** Take a normal FP32 model and store its weights in **8-bit integers** instead of
32-bit floats. Nothing about the network changes except *how the numbers are encoded*, yet
the model gets **~4x smaller** on disk for almost no accuracy loss.

The parts to focus on are:

1. **The quantization formula** - how a float maps to an integer and back.
2. **The compression ratio** - bits before / bits after.

At the end we **record size, accuracy and latency before vs. after** so the trade-off is concrete:

| model | size | accuracy | latency |
|-------|------|----------|---------|
| FP32 baseline | large | high | baseline |
| **INT8 (quantized)** | **~4x smaller** | ~same | see note |

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.
Full run is a few minutes on a Colab GPU.

## 1. Setup

On Colab `torch`/`torchvision` are pre-installed, so nothing to `pip install`.

In [ ]:
import io, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- knobs you can play with ---
EPOCHS     = 12      # epochs to train the FP32 baseline
BATCH_SIZE = 128
LR         = 0.05
N_BITS     = 8       # quantize weights to this many bits (try 4, 6, 8)

## 2. Data - MNIST

10 classes (handwritten digits 0-9) of 28x28 grayscale images. Small and quick to download.

In [ ]:
mean = (0.1307,)
std  = (0.3081,)

train_tf = T.Compose([
    T.RandomCrop(28, padding=2),
    T.ToTensor(),
    T.Normalize(mean, std),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False, num_workers=2)

classes = train_set.classes
print(len(train_set), "train /", len(test_set), "test images")

## 3. The model

A plain VGG-style CNN. Nothing special - it is just the FP32 network we will compress.
Almost all of its size lives in the conv/linear **weights**, which is exactly what we will
quantize.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

class Net(nn.Module):               # ~1.1M params
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 64),   conv_block(64, 64),   nn.MaxPool2d(2),   # 28 -> 14
            conv_block(64, 128), conv_block(128, 128), nn.MaxPool2d(2),   # 14 -> 7
            conv_block(128, 256),conv_block(256, 256), nn.MaxPool2d(2),   # 7  -> 3
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"params: {count_params(Net()):,}")

## 4. Train / eval / metric utilities

Plain training and evaluation loops, a single-image latency probe (batch size 1 on
**CPU**, the realistic edge setting), and an on-disk model-size probe.

In [ ]:
def train(model, epochs=EPOCHS, tag=""):
    model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        acc = evaluate(model)
        print(f"[{tag}] epoch {ep+1:02d}/{epochs}  test_acc={acc:.2f}%")
    return model

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    model.train()
    return 100.0 * correct / total

@torch.no_grad()
def latency_ms(model, n=100):
    """Average single-image inference latency on CPU (ms)."""
    model = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 1, 28, 28)
    for _ in range(10):            # warmup
        model(x)
    t0 = time.time()
    for _ in range(n):
        model(x)
    return (time.time() - t0) / n * 1000.0

def model_size_mb(state_dict):
    """On-disk size of a saved state dict (MB)."""
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getbuffer().nbytes / 1e6

## 5. Train the FP32 baseline

Train the network normally. This is the **"before"** model - full 32-bit floats.

In [ ]:
model_fp32 = train(Net(), tag="fp32")
fp32_acc = evaluate(model_fp32)
print(f"\nFP32 baseline accuracy: {fp32_acc:.2f}%")

## 6. The quantization scheme & formula 🔑

This is the heart of the exercise. A real number $x$ is mapped to a small integer $q$ by a
**scale** and an integer **zero-point**:

$$
q \;=\; \mathrm{round}\!\left(\frac{x}{\text{scale}}\right) + \text{zero\_point},
\qquad\qquad
\hat{x} \;=\; \text{scale}\cdot\big(q - \text{zero\_point}\big)
$$

$q$ is clamped to the integer range, e.g. **INT8** is $[-128, 127]$. $\hat{x}$ is the
**dequantized** value - close to $x$ but not exact; the gap is the *quantization error*.

Two choices define a scheme:

- **Symmetric vs. affine.** *Symmetric* forces `zero_point = 0` and uses
  $\text{scale} = \max(|x|)/q_{\max}$ - the range is centered on 0 (natural for weights).
  *Affine* fits the true $[\min, \max]$ with a non-zero `zero_point` (better for skewed
  data like post-ReLU activations).
- **Per-tensor vs. per-channel.** *Per-tensor* uses one scale for the whole weight tensor.
  *Per-channel* uses one scale **per output channel** - a few outlier channels no longer
  blow up everyone else's range, so the error is smaller.

The **compression ratio** is just the bit ratio: FP32 -> INT8 is $32/8 = $ **4x** fewer bits.

In [ ]:
def quantize(x, n_bits=8, symmetric=True, per_channel=False):
    """Float tensor -> (int codes q, scale, zero_point). Channel axis = dim 0."""
    qmin, qmax = -(2 ** (n_bits - 1)), 2 ** (n_bits - 1) - 1   # e.g. INT8: [-128, 127]

    if per_channel:                       # reduce over every dim except output channels
        flat = x.flatten(1)
        xmin, xmax = flat.min(1).values, flat.max(1).values
    else:
        xmin, xmax = x.min(), x.max()

    if symmetric:
        # TODO: symmetric scale = max(|x|) / qmax, zero_point stays 0; use .clamp(min=1e-8)
        scale = torch.ones_like(xmax)
        zero_point = torch.zeros_like(scale)
    else:                                 # affine: fit the real [min, max]
        # TODO: affine scale = (xmax - xmin) / (qmax - qmin); zero_point = qmin - round(xmin / scale)
        scale = torch.ones_like(xmax)
        zero_point = torch.zeros_like(scale)

    shape = [-1] + [1] * (x.dim() - 1)    # broadcast per-channel scale over the tensor
    s = scale.view(shape) if per_channel else scale
    z = zero_point.view(shape) if per_channel else zero_point

    # TODO: quantize with q = round(x / s) + z, then clamp into [qmin, qmax]
    q = torch.clamp(x, qmin, qmax)
    return q, scale, zero_point

def dequantize(q, scale, zero_point, per_channel=False):
    if per_channel:
        shape = [-1] + [1] * (q.dim() - 1)
        scale, zero_point = scale.view(shape), zero_point.view(shape)
    return scale * (q - zero_point)

# --- see it on one real conv weight: per-tensor vs per-channel error ---
w = model_fp32.features[0][0].weight.data.cpu()    # first conv layer
for pc in (False, True):
    q, s, z = quantize(w, n_bits=N_BITS, symmetric=True, per_channel=pc)
    err = F.mse_loss(dequantize(q, s, z, per_channel=pc), w).item()
    print(f"{'per-channel' if pc else 'per-tensor ':12}  INT{N_BITS}  MSE = {err:.3e}")

print(f"\\ncompression ratio: 32 / {N_BITS} = {32 / N_BITS:.0f}x fewer bits per weight")

## 7. What quantization does to the weights

The same weights, before and after. Quantization snaps every float onto one of a few
evenly spaced levels - that grid is the whole story. Per-channel keeps the grid tight for
each output channel.

In [ ]:
import matplotlib.pyplot as plt

q, s, z = quantize(w, n_bits=N_BITS, symmetric=True, per_channel=True)
w_hat = dequantize(q, s, z, per_channel=True)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(w.flatten().numpy(), bins=80, color="#888")
ax[0].set_title(f"original FP32 weights  ({w.numel()} values)")
ax[1].hist(w_hat.flatten().numpy(), bins=80, color="#5cb85c")
ax[1].set_title(f"after INT{N_BITS} quantize -> dequantize (discrete levels)")
for a in ax: a.set_xlabel("weight value")
plt.tight_layout(); plt.show()

print(f"int codes are integers in [{q.min():.0f}, {q.max():.0f}] - stored as int{N_BITS}")

## 8. Post-training quantization: quantize every weight

Now apply the formula to the **whole model** - this is *post-training quantization (PTQ)*:
take the trained FP32 weights and convert each conv/linear weight to INT8 (symmetric,
per-channel). No retraining.

We build two views of the quantized model:

- a **packed** state dict (int8 codes + per-channel scales) -> the real **on-disk size**;
- a **dequantized** copy (weights replaced by `w_hat`) -> run it to read the real
  **accuracy** the int8 weights give.

In [ ]:
QUANTIZABLE = (nn.Conv2d, nn.Linear)

def quantize_model(model, n_bits=8, per_channel=True):
    """Return (packed_state_dict, dequantized_model) for weight-only PTQ."""
    packed = {}
    deq = copy.deepcopy(model).cpu().eval()
    for name, mod in deq.named_modules():
        if isinstance(mod, QUANTIZABLE):
            q, s, z = quantize(mod.weight.data, n_bits=n_bits, symmetric=True, per_channel=per_channel)
            packed[name + ".weight_int"]   = q.to(torch.int8)
            packed[name + ".weight_scale"] = s.clone()
            mod.weight.data = dequantize(q, s, z, per_channel=per_channel)   # fake-quant
    # keep everything else (biases, BatchNorm stats) in fp32
    for k, v in deq.state_dict().items():
        if not any(k == n + ".weight" for n, m in deq.named_modules() if isinstance(m, QUANTIZABLE)):
            packed[k] = v
    return packed, deq

packed_pc, model_int8 = quantize_model(model_fp32, n_bits=N_BITS, per_channel=True)
_,         model_int8_pt = quantize_model(model_fp32, n_bits=N_BITS, per_channel=False)

int8_acc    = evaluate(model_int8.to(device))
int8_acc_pt = evaluate(model_int8_pt.to(device))
print(f"accuracy  FP32         : {fp32_acc:.2f}%")
print(f"accuracy  INT8 per-tensor : {int8_acc_pt:.2f}%")
print(f"accuracy  INT8 per-channel: {int8_acc:.2f}%   <- tighter grid, less drop")

## 9. Results - size, accuracy & latency, before vs. after

The payoff. **Size** is the guaranteed win and the headline metric for quantization.
Accuracy barely moves. Latency needs a word - see the note below the chart.

In [ ]:
fp32_size = model_size_mb(model_fp32.state_dict())
int8_size = model_size_mb(packed_pc)

rows = [
    ("FP32 baseline", fp32_size, fp32_acc, latency_ms(model_fp32)),
    ("INT8 (per-channel)", int8_size, int8_acc, latency_ms(model_int8)),
]

print(f"{'model':<22}{'size':>10}{'accuracy':>11}{'CPU latency':>14}")
print("-" * 57)
for name, sz, acc, lat in rows:
    print(f"{name:<22}{sz:>8.2f}MB{acc:>10.2f}%{lat:>12.2f}ms")

print(f"\n>>> {fp32_size/int8_size:.1f}x smaller on disk for {fp32_acc-int8_acc:+.2f}% accuracy.")

# --- plots ---
names  = [r[0] for r in rows]
sizes  = [r[1] for r in rows]
accs   = [r[2] for r in rows]
lats   = [r[3] for r in rows]
colors = ["#d9534f", "#5cb85c"]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(names, sizes, color=colors); ax[0].set_title("Model size (MB)  <- the win")
ax[1].bar(names, accs,  color=colors); ax[1].set_title("Accuracy (%)"); ax[1].set_ylim(min(accs)-3, 100)
ax[2].bar(names, lats,  color=colors); ax[2].set_title("CPU latency / image (ms)")
for a in ax: a.tick_params(axis="x", rotation=10)
plt.tight_layout(); plt.show()

## 10. Takeaways & things to try

**What you saw:** storing weights as INT8 makes the model **~4x smaller** with almost no
accuracy loss, and **per-channel** scales keep the error lower than per-tensor.

**A note on latency.** The size win is real and guaranteed. The *speed* win is **not** shown
here: our quantized model dequantizes back to FP32 and runs the ordinary float kernels, so
its latency matches the baseline. A real latency speedup needs **integer kernels** that do
the matmul directly in INT8 - that is the job of inference runtimes like `llama.cpp` /
`vla.cpp`, not of the math in this notebook. Quantization's portable, always-true promise is
**memory**; speed depends on hardware and kernel support.

**Experiment (edit and re-run):**
- Drop `N_BITS` to `4` or `6`. How far can you go before accuracy falls off a cliff?
- Compare **per-tensor vs. per-channel** at low bit-widths - the gap grows.
- Switch the weight scheme to **affine** (`symmetric=False`) - for zero-centered weights it
  barely helps; it matters far more for post-ReLU activations.
- Quantize **activations** too (not just weights): that is what unlocks full integer kernels.

**Why it matters for the edge:** quantization is the most widely deployed compression
technique - it is how billion-parameter models fit on phones and laptops. It composes with
the other exercises: **distill**, then **prune**, then **quantize** the result.